# Experiment 7 — Autoencoders, Convolutional AE, Denoising AE, VAE
### CS3807 — Deep Learning Laboratory

This notebook implements, trains and evaluates all four models required by
Experiment 7 (Fully Connected Autoencoder, Convolutional Autoencoder,
Denoising Convolutional Autoencoder, Variational Autoencoder) on MNIST,
and produces **all 10 plots required under Section 22** (as EPS files at
600 DPI) plus **every numeric table** required elsewhere in the report
(Sections 9, 11, 14, 21, 24, 25, 27).

All plots are saved to `/content/plots/` as `.eps` files at 600 DPI, and are
also displayed inline. All tables are printed as pandas DataFrames (and also
saved as CSV files in `/content/tables/`) so you can paste the exact numbers
from **your own run** into the LaTeX report, as required by the experiment
brief.

> Run all cells top to bottom. Total runtime on a Colab GPU runtime is
> roughly 8-12 minutes.

In [ ]:
# ============================================================
# CELL 1 — Setup, imports, reproducibility, global config
# ============================================================
!pip -q install scikit-image --upgrade

import os, time, json, random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
import pandas as pd
from skimage.metrics import structural_similarity as ssim_metric

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PLOT_DIR = "/content/plots"
TABLE_DIR = "/content/tables"
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

# All figures are saved as vector/raster EPS at 600 DPI as required.
matplotlib.rcParams['savefig.dpi'] = 600
matplotlib.rcParams['figure.dpi'] = 100

def save_eps(fig, name):
    # Save a matplotlib figure as EPS at 600 DPI into PLOT_DIR.
    path = os.path.join(PLOT_DIR, name if name.endswith(".eps") else name + ".eps")
    fig.savefig(path, format='eps', dpi=600, bbox_inches='tight')
    print(f"Saved: {path}")
    return path

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ============================================================
# CELL 2 — Load MNIST, build the recommended laboratory subset
# (10,000 train images / 2,000 test images), normalize to [0,1]
# ============================================================
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.mnist.load_data()

N_TRAIN = 10000
N_TEST  = 2000

rng = np.random.RandomState(SEED)
train_idx = rng.choice(len(x_train_full), N_TRAIN, replace=False)
test_idx  = rng.choice(len(x_test_full),  N_TEST,  replace=False)

x_train = x_train_full[train_idx].astype("float32") / 255.0
y_train = y_train_full[train_idx]
x_test  = x_test_full[test_idx].astype("float32") / 255.0
y_test  = y_test_full[test_idx]

# Convolutional / VAE shape: (N, 28, 28, 1)
x_train_img = x_train.reshape(-1, 28, 28, 1)
x_test_img  = x_test.reshape(-1, 28, 28, 1)

# Fully-connected shape: (N, 784)
x_train_flat = x_train.reshape(-1, 784)
x_test_flat  = x_test.reshape(-1, 784)

print("Train images:", x_train_img.shape, " Test images:", x_test_img.shape)

In [ ]:
# ============================================================
# CELL 3 — Metric helpers (MSE, MAE, SSIM) used throughout
# ============================================================
def compute_mse(x, x_hat):
    return float(np.mean((x - x_hat) ** 2))

def compute_mae(x, x_hat):
    return float(np.mean(np.abs(x - x_hat)))

def compute_mean_ssim(x, x_hat):
    # x, x_hat: (N,28,28) or (N,28,28,1) arrays in [0,1].
    x = x.reshape(-1, 28, 28)
    x_hat = x_hat.reshape(-1, 28, 28)
    vals = [ssim_metric(x[i], x_hat[i], data_range=1.0) for i in range(x.shape[0])]
    return float(np.mean(vals))

def per_image_mse(x, x_hat):
    x = x.reshape(x.shape[0], -1)
    x_hat = x_hat.reshape(x_hat.shape[0], -1)
    return np.mean((x - x_hat) ** 2, axis=1)

In [ ]:
# ============================================================
# CELL 4 — SECTION 7: Fully Connected Autoencoder (784-128-32-16-32-128-784)
# ============================================================
LATENT_DIM_FC = 16

def build_fc_autoencoder(latent_dim=16):
    inp = layers.Input(shape=(784,), name="input_784")
    x = layers.Dense(128, activation='relu')(inp)
    x = layers.Dense(32, activation='relu')(x)
    z = layers.Dense(latent_dim, activation='relu', name="latent")(x)
    x = layers.Dense(32, activation='relu')(z)
    x = layers.Dense(128, activation='relu')(x)
    out = layers.Dense(784, activation='sigmoid')(x)
    autoencoder = models.Model(inp, out, name=f"FC_Autoencoder_z{latent_dim}")
    autoencoder.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                         loss='binary_crossentropy')
    return autoencoder

fc_ae = build_fc_autoencoder(LATENT_DIM_FC)
fc_ae.summary()

t0 = time.time()
fc_history = fc_ae.fit(
    x_train_flat, x_train_flat,
    validation_data=(x_test_flat, x_test_flat),
    epochs=20, batch_size=128, shuffle=True, verbose=2
)
fc_train_time = time.time() - t0
print(f"FC-AE training time: {fc_train_time:.2f} s")

fc_recon_flat = fc_ae.predict(x_test_flat, verbose=0)
fc_recon_img = fc_recon_flat.reshape(-1, 28, 28, 1)

fc_mse  = compute_mse(x_test_flat, fc_recon_flat)
fc_mae  = compute_mae(x_test_flat, fc_recon_flat)
fc_ssim = compute_mean_ssim(x_test_img, fc_recon_img)
fc_params = fc_ae.count_params()

print(f"FC-AE  -> MSE={fc_mse:.6f}  MAE={fc_mae:.6f}  SSIM={fc_ssim:.4f}  Params={fc_params}")

In [ ]:
# ============================================================
# CELL 5 — PLOT 1 (Section 22, item 1): Original vs Reconstructed (FC-AE)
# ============================================================
n_show = 10
fig, axes = plt.subplots(2, n_show, figsize=(1.6 * n_show, 3.4))
for i in range(n_show):
    axes[0, i].imshow(x_test_img[i].squeeze(), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(fc_recon_img[i].squeeze(), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Reconstructed", fontsize=10)
fig.suptitle("Plot 1: Original vs Reconstructed Images (FC-Autoencoder)")
fig.text(0.06, 0.72, "Original", rotation=90, va='center', fontsize=11)
fig.text(0.06, 0.30, "Reconstructed", rotation=90, va='center', fontsize=11)
plt.tight_layout(rect=[0.06, 0, 1, 0.94])
save_eps(fig, "plot1_original_vs_reconstructed.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 6 — PLOT 2 (Section 22, item 2): Training/Validation loss, basic AE
# ============================================================
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(fc_history.history['loss'], label='Training loss')
ax.plot(fc_history.history['val_loss'], label='Validation loss')
ax.set_xlabel("Epoch")
ax.set_ylabel("Reconstruction loss (BCE)")
ax.set_title("Plot 2: Training/Validation Loss (FC-Autoencoder)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_eps(fig, "plot2_fcae_training_validation_loss.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 7 — SECTION 9 table: FC-AE reconstruction metrics
# ============================================================
sec9_table = pd.DataFrame({
    "Metric": ["Test MSE", "Test MAE", "Mean SSIM"],
    "Value":  [fc_mse, fc_mae, fc_ssim]
})
print("Section 9 - Reconstruction Metrics (Fully Connected Autoencoder)")
display(sec9_table)
sec9_table.to_csv(f"{TABLE_DIR}/section9_fcae_metrics.csv", index=False)

In [ ]:
# ============================================================
# CELL 8 — SECTION 10: Convolutional Autoencoder
# ============================================================
def build_conv_autoencoder():
    inp = layers.Input(shape=(28, 28, 1), name="input_28x28x1")
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D(2, padding='same')(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)
    latent = layers.Conv2D(64, 3, activation='relu', padding='same', name="latent_feature_map")(x)
    x = layers.UpSampling2D(2)(latent)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.UpSampling2D(2)(x)
    out = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)
    cae = models.Model(inp, out, name="Convolutional_Autoencoder")
    cae.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy')
    return cae

cae = build_conv_autoencoder()
cae.summary()

t0 = time.time()
cae_history = cae.fit(
    x_train_img, x_train_img,
    validation_data=(x_test_img, x_test_img),
    epochs=20, batch_size=128, shuffle=True, verbose=2
)
cae_train_time = time.time() - t0
print(f"CAE training time: {cae_train_time:.2f} s")

cae_recon_img = cae.predict(x_test_img, verbose=0)

cae_mse  = compute_mse(x_test_img, cae_recon_img)
cae_mae  = compute_mae(x_test_img, cae_recon_img)
cae_ssim = compute_mean_ssim(x_test_img, cae_recon_img)
cae_params = cae.count_params()

print(f"CAE -> MSE={cae_mse:.6f}  MAE={cae_mae:.6f}  SSIM={cae_ssim:.4f}  Params={cae_params}")

In [ ]:
# ============================================================
# CELL 9 — PLOT 3 (Section 22, item 3): FC-AE vs CAE reconstruction
# ============================================================
n_show = 10
fig, axes = plt.subplots(3, n_show, figsize=(1.6 * n_show, 5.0))
for i in range(n_show):
    axes[0, i].imshow(x_test_img[i].squeeze(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(fc_recon_img[i].squeeze(), cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(cae_recon_img[i].squeeze(), cmap='gray'); axes[2, i].axis('off')
fig.text(0.06, 0.82, "Original", rotation=90, va='center', fontsize=11)
fig.text(0.06, 0.50, "FC-AE", rotation=90, va='center', fontsize=11)
fig.text(0.06, 0.18, "Conv-AE", rotation=90, va='center', fontsize=11)
fig.suptitle("Plot 3: Original vs FC-AE vs Convolutional-AE Reconstruction")
plt.tight_layout(rect=[0.06, 0, 1, 0.94])
save_eps(fig, "plot3_fc_vs_conv_reconstruction.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 10 — SECTION 11 table: FC-AE vs Conv-AE comparison
# ============================================================
sec11_table = pd.DataFrame({
    "Model":      ["Fully Connected AE", "Convolutional AE"],
    "MSE":        [fc_mse, cae_mse],
    "MAE":        [fc_mae, cae_mae],
    "SSIM":       [fc_ssim, cae_ssim],
    "Parameters": [fc_params, cae_params]
})
print("Section 11 - Comparing Fully Connected and Convolutional Autoencoders")
display(sec11_table)
sec11_table.to_csv(f"{TABLE_DIR}/section11_fc_vs_conv.csv", index=False)

In [ ]:
# ============================================================
# CELL 11 — SECTION 12/13: Noise functions + Denoising Convolutional AE
# Trained at sigma = 0.2 (middle of the suggested {0.1, 0.2, 0.3} range).
# ============================================================
def add_gaussian_noise(images, sigma, seed=None):
    rs = np.random.RandomState(seed)
    noisy = images + rs.normal(0.0, sigma, images.shape).astype("float32")
    return np.clip(noisy, 0.0, 1.0)

def add_salt_and_pepper_noise(images, p, seed=None):
    rs = np.random.RandomState(seed)
    noisy = images.copy()
    mask = rs.rand(*images.shape)
    noisy[mask < (p / 2)] = 0.0
    noisy[(mask >= (p / 2)) & (mask < p)] = 1.0
    return noisy

TRAIN_NOISE_SIGMA = 0.2
x_train_noisy = add_gaussian_noise(x_train_img, TRAIN_NOISE_SIGMA, seed=SEED)
x_test_noisy  = add_gaussian_noise(x_test_img,  TRAIN_NOISE_SIGMA, seed=SEED + 1)

denoising_cae = build_conv_autoencoder()
denoising_cae._name = "Denoising_Convolutional_Autoencoder"
denoising_cae.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss='binary_crossentropy')

t0 = time.time()
dcae_history = denoising_cae.fit(
    x_train_noisy, x_train_img,
    validation_data=(x_test_noisy, x_test_img),
    epochs=20, batch_size=128, shuffle=True, verbose=2
)
dcae_train_time = time.time() - t0
print(f"Denoising CAE training time: {dcae_train_time:.2f} s")

dcae_recon_img = denoising_cae.predict(x_test_noisy, verbose=0)
dcae_mse  = compute_mse(x_test_img, dcae_recon_img)
dcae_mae  = compute_mae(x_test_img, dcae_recon_img)
dcae_ssim = compute_mean_ssim(x_test_img, dcae_recon_img)
dcae_params = denoising_cae.count_params()

print(f"Denoising CAE -> MSE={dcae_mse:.6f}  MAE={dcae_mae:.6f}  SSIM={dcae_ssim:.4f}  Params={dcae_params}")

In [ ]:
# ============================================================
# CELL 12 — PLOT 4 (Section 22, item 4): Clean vs Noisy vs Denoised
# ============================================================
n_show = 10
fig, axes = plt.subplots(3, n_show, figsize=(1.6 * n_show, 5.0))
for i in range(n_show):
    axes[0, i].imshow(x_test_img[i].squeeze(), cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_test_noisy[i].squeeze(), cmap='gray'); axes[1, i].axis('off')
    axes[2, i].imshow(dcae_recon_img[i].squeeze(), cmap='gray'); axes[2, i].axis('off')
fig.text(0.06, 0.82, "Clean", rotation=90, va='center', fontsize=11)
fig.text(0.06, 0.50, "Noisy", rotation=90, va='center', fontsize=11)
fig.text(0.06, 0.18, "Denoised", rotation=90, va='center', fontsize=11)
fig.suptitle(f"Plot 4: Clean vs Noisy (sigma={TRAIN_NOISE_SIGMA}) vs Denoised Reconstruction")
plt.tight_layout(rect=[0.06, 0, 1, 0.94])
save_eps(fig, "plot4_clean_noisy_denoised.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 13 — PLOT 5 (Section 22, item 5) + SECTION 14 table:
# Noise level vs MSE / MAE / SSIM, using the denoising model trained above
# evaluated against test images corrupted at sigma = 0.1, 0.2, 0.3.
# ============================================================
noise_levels = [0.1, 0.2, 0.3]
noise_results = []
for sigma in noise_levels:
    x_noisy_eval = add_gaussian_noise(x_test_img, sigma, seed=SEED + 100)
    recon = denoising_cae.predict(x_noisy_eval, verbose=0)
    noise_results.append({
        "Noise Level": sigma,
        "MSE":  compute_mse(x_test_img, recon),
        "MAE":  compute_mae(x_test_img, recon),
        "SSIM": compute_mean_ssim(x_test_img, recon)
    })

sec14_table = pd.DataFrame(noise_results)
print("Section 14 - Noise Level vs Reconstruction Quality")
display(sec14_table)
sec14_table.to_csv(f"{TABLE_DIR}/section14_noise_vs_quality.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(sec14_table["Noise Level"], sec14_table["MSE"], marker='o', color='tab:red')
axes[0].set_xlabel("Noise level (sigma)"); axes[0].set_ylabel("MSE"); axes[0].set_title("MSE vs Noise Level")
axes[1].plot(sec14_table["Noise Level"], sec14_table["MAE"], marker='o', color='tab:orange')
axes[1].set_xlabel("Noise level (sigma)"); axes[1].set_ylabel("MAE"); axes[1].set_title("MAE vs Noise Level")
axes[2].plot(sec14_table["Noise Level"], sec14_table["SSIM"], marker='o', color='tab:green')
axes[2].set_xlabel("Noise level (sigma)"); axes[2].set_ylabel("SSIM"); axes[2].set_title("SSIM vs Noise Level")
for a in axes:
    a.grid(alpha=0.3)
fig.suptitle("Plot 5: Noise Level vs Reconstruction Quality (separate axes)")
plt.tight_layout(rect=[0, 0, 1, 0.93])
save_eps(fig, "plot5_noise_level_vs_quality.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 14 — SECTION 21 table (part 1): 3-model reconstruction comparison
# ============================================================
sec21_recon_table = pd.DataFrame({
    "Model":      ["Fully Connected AE", "Convolutional AE", "Denoising CAE"],
    "MSE":        [fc_mse, cae_mse, dcae_mse],
    "MAE":        [fc_mae, cae_mae, dcae_mae],
    "SSIM":       [fc_ssim, cae_ssim, dcae_ssim],
    "Parameters": [fc_params, cae_params, dcae_params]
})
print("Section 21 - Reconstruction Comparison Across Models")
display(sec21_recon_table)
sec21_recon_table.to_csv(f"{TABLE_DIR}/section21_model_comparison.csv", index=False)

In [ ]:
# ============================================================
# CELL 15 — SECTION 15-17: Variational Autoencoder (2-D latent space)
# ============================================================
LATENT_DIM_VAE = 2

class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# ---- Encoder ----
enc_inp = layers.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(enc_inp)
x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(16, activation='relu')(x)
z_mean = layers.Dense(LATENT_DIM_VAE, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM_VAE, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
vae_encoder = models.Model(enc_inp, [z_mean, z_log_var, z], name="VAE_Encoder")
vae_encoder.summary()

# ---- Decoder ----
dec_inp = layers.Input(shape=(LATENT_DIM_VAE,))
x = layers.Dense(7 * 7 * 64, activation='relu')(dec_inp)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
dec_out = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)
vae_decoder = models.Model(dec_inp, dec_out, name="VAE_Decoder")
vae_decoder.summary()

class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        return self.decoder(z)

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x)
            reconstruction = self.decoder(z)
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(losses.binary_crossentropy(x, reconstruction), axis=(1, 2))
            )
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
            )
            total_loss = recon_loss + kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        z_mean, z_log_var, z = self.encoder(x)
        reconstruction = self.decoder(z)
        recon_loss = tf.reduce_mean(
            tf.reduce_sum(losses.binary_crossentropy(x, reconstruction), axis=(1, 2))
        )
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
        )
        total_loss = recon_loss + kl_loss
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

vae = VAE(vae_encoder, vae_decoder)
vae.compile(optimizer=optimizers.Adam(learning_rate=1e-3))

t0 = time.time()
vae_history = vae.fit(
    x_train_img, x_train_img,
    validation_data=(x_test_img, x_test_img),
    epochs=30, batch_size=128, shuffle=True, verbose=2
)
vae_train_time = time.time() - t0
print(f"VAE training time: {vae_train_time:.2f} s")

In [ ]:
# ============================================================
# CELL 16 — PLOT 9 (Section 22, item 9): VAE train/val loss + Section 21 VAE table
# ============================================================
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(vae_history.history['loss'], label='Training loss (total)')
ax.plot(vae_history.history['val_loss'], label='Validation loss (total)')
ax.set_xlabel("Epoch")
ax.set_ylabel("VAE loss (reconstruction + KL)")
ax.set_title("Plot 9: Training/Validation Reconstruction Loss (VAE)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_eps(fig, "plot9_vae_training_validation_loss.eps")
plt.show()

# Final-epoch VAE metric breakdown
vae_final_recon_loss = float(vae_history.history['reconstruction_loss'][-1])
vae_final_kl_loss    = float(vae_history.history['kl_loss'][-1])
vae_final_total_loss = float(vae_history.history['loss'][-1])

# Test-set reconstruction metrics: use the encoder's mean (z_mean) — the
# deterministic reconstruction — as the standard evaluation reconstruction.
z_mean_test, z_log_var_test, z_test = vae_encoder.predict(x_test_img, verbose=0)
vae_recon_img = vae_decoder.predict(z_mean_test, verbose=0)

vae_test_mse  = compute_mse(x_test_img, vae_recon_img)
vae_test_mae  = compute_mae(x_test_img, vae_recon_img)
vae_test_ssim = compute_mean_ssim(x_test_img, vae_recon_img)
vae_params = vae_encoder.count_params() + vae_decoder.count_params()

sec21_vae_table = pd.DataFrame({
    "VAE Metric": ["Reconstruction Loss", "KL Loss", "Total Loss",
                    "Test MSE", "Test MAE", "Mean SSIM"],
    "Value": [vae_final_recon_loss, vae_final_kl_loss, vae_final_total_loss,
              vae_test_mse, vae_test_mae, vae_test_ssim]
})
print("Section 21 - VAE Metrics")
display(sec21_vae_table)
sec21_vae_table.to_csv(f"{TABLE_DIR}/section21_vae_metrics.csv", index=False)
print(f"VAE parameter count (encoder+decoder): {vae_params}")

In [ ]:
# ============================================================
# CELL 17 — PLOT 6 (Section 22, item 6): VAE latent-space visualization
# ============================================================
fig, ax = plt.subplots(figsize=(6.5, 5.5))
scatter = ax.scatter(z_mean_test[:, 0], z_mean_test[:, 1], c=y_test, cmap='tab10', s=8, alpha=0.8)
legend1 = ax.legend(*scatter.legend_elements(), title="Digit", loc="center left",
                     bbox_to_anchor=(1.02, 0.5))
ax.add_artist(legend1)
ax.set_xlabel("$z_1$")
ax.set_ylabel("$z_2$")
ax.set_title("Plot 6: VAE Latent-Space Visualization (Test Set)")
ax.grid(alpha=0.3)
plt.tight_layout()
save_eps(fig, "plot6_vae_latent_space.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 18 — PLOT 7 (Section 22, item 7): Randomly generated VAE images
# ============================================================
n_gen = 25
grid_side = 5
z_samples = np.random.RandomState(SEED).normal(size=(n_gen, LATENT_DIM_VAE)).astype("float32")
generated = vae_decoder.predict(z_samples, verbose=0)

fig, axes = plt.subplots(grid_side, grid_side, figsize=(7, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].squeeze(), cmap='gray')
    ax.axis('off')
fig.suptitle("Plot 7: Randomly Generated VAE Images (z ~ N(0, I))")
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_eps(fig, "plot7_vae_generated_images.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 19 — PLOT 8 (Section 22, item 8): Latent-space interpolation
# ============================================================
# Pick two test digits from different classes as z_A, z_B
idx_A = np.where(y_test == 3)[0][0]
idx_B = np.where(y_test == 8)[0][0]
z_A = z_mean_test[idx_A]
z_B = z_mean_test[idx_B]

alphas = np.arange(0.0, 1.0001, 0.1)
z_interp = np.array([(1 - a) * z_A + a * z_B for a in alphas], dtype="float32")
interp_images = vae_decoder.predict(z_interp, verbose=0)

fig, axes = plt.subplots(1, len(alphas), figsize=(1.6 * len(alphas), 2.0))
for i, ax in enumerate(axes):
    ax.imshow(interp_images[i].squeeze(), cmap='gray')
    ax.axis('off')
    ax.set_title(f"{alphas[i]:.1f}", fontsize=8)
fig.suptitle("Plot 8: Latent-Space Interpolation ($z_A$ = digit 3 -> $z_B$ = digit 8)")
plt.tight_layout(rect=[0, 0, 1, 0.85])
save_eps(fig, "plot8_latent_space_interpolation.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 20 — PLOT 10 (Section 22, item 10): Reconstruction-error distribution
# on the test set (Section 23), + Section 24 high-error image table.
# Computed for the Fully Connected Autoencoder ("basic autoencoder").
# ============================================================
errors = per_image_mse(x_test_flat, fc_recon_flat)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.hist(errors, bins=40, color='tab:blue', alpha=0.8, edgecolor='black')
ax.set_xlabel("Per-image reconstruction error (MSE)")
ax.set_ylabel("Number of test images")
ax.set_title("Plot 10: Reconstruction-Error Distribution (Test Set, FC-AE)")
ax.grid(alpha=0.3)
plt.tight_layout()
save_eps(fig, "plot10_reconstruction_error_distribution.eps")
plt.show()

print(f"Reconstruction error: min={errors.min():.6f}  max={errors.max():.6f}  "
      f"mean={errors.mean():.6f}  std={errors.std():.6f}")

# ---- Section 24: five highest-error test images ----
top5_idx = np.argsort(errors)[-5:][::-1]

fig, axes = plt.subplots(5, 2, figsize=(3.4, 8.5))
for row, idx in enumerate(top5_idx):
    axes[row, 0].imshow(x_test_img[idx].squeeze(), cmap='gray'); axes[row, 0].axis('off')
    axes[row, 1].imshow(fc_recon_img[idx].squeeze(), cmap='gray'); axes[row, 1].axis('off')
axes[0, 0].set_title("Original", fontsize=10)
axes[0, 1].set_title("Reconstruction", fontsize=10)
fig.suptitle("Section 24: Five Highest-Error Test Images", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
save_eps(fig, "plot_extra_section24_high_error_images.eps")
plt.show()

sec24_table = pd.DataFrame({
    "Sample": [1, 2, 3, 4, 5],
    "Test Index": top5_idx,
    "True Digit": y_test[top5_idx],
    "Reconstruction Error (MSE)": errors[top5_idx]
})
print("Section 24 - Required High-Error Image Analysis")
display(sec24_table)
sec24_table.to_csv(f"{TABLE_DIR}/section24_high_error_images.csv", index=False)

In [ ]:
# ============================================================
# CELL 21 — SECTION 25 table: Consolidated Results (all 4 models)
# ============================================================
sec25_table = pd.DataFrame({
    "Model":      ["FC Autoencoder", "Conv. Autoencoder", "Denoising CAE", "VAE"],
    "MSE":        [fc_mse, cae_mse, dcae_mse, vae_test_mse],
    "MAE":        [fc_mae, cae_mae, dcae_mae, vae_test_mae],
    "SSIM":       [fc_ssim, cae_ssim, dcae_ssim, vae_test_ssim],
    "Parameters": [fc_params, cae_params, dcae_params, vae_params],
    "Time (s)":   [fc_train_time, cae_train_time, dcae_train_time, vae_train_time]
})
print("Section 25 - Consolidated Results")
display(sec25_table)
sec25_table.to_csv(f"{TABLE_DIR}/section25_consolidated_results.csv", index=False)

In [ ]:
# ============================================================
# CELL 22 — SECTION 27: Additional latent-dimension study (FC-AE),
# d_z in {2, 8, 16, 32} + bonus Plot 11 (Latent Dimension vs MSE)
# ============================================================
latent_dims = [2, 8, 16, 32]
latent_results = []

for dz in latent_dims:
    print(f"\\n--- Training FC-AE with latent dimension = {dz} ---")
    model_dz = build_fc_autoencoder(latent_dim=dz)
    model_dz.fit(
        x_train_flat, x_train_flat,
        validation_data=(x_test_flat, x_test_flat),
        epochs=20, batch_size=128, shuffle=True, verbose=0
    )
    recon_dz = model_dz.predict(x_test_flat, verbose=0)
    recon_dz_img = recon_dz.reshape(-1, 28, 28, 1)
    mse_dz  = compute_mse(x_test_flat, recon_dz)
    ssim_dz = compute_mean_ssim(x_test_img, recon_dz_img)
    params_dz = model_dz.count_params()
    latent_results.append({"Latent Dimension": dz, "MSE": mse_dz,
                            "SSIM": ssim_dz, "Parameters": params_dz})
    print(f"d_z={dz}: MSE={mse_dz:.6f}  SSIM={ssim_dz:.4f}  Params={params_dz}")

sec27_table = pd.DataFrame(latent_results)
print("\\nSection 27 - Additional Latent-Dimension Study")
display(sec27_table)
sec27_table.to_csv(f"{TABLE_DIR}/section27_latent_dimension_study.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.plot(sec27_table["Latent Dimension"], sec27_table["MSE"], marker='o', color='tab:purple')
ax.set_xlabel("Latent dimension $d_z$")
ax.set_ylabel("Test MSE")
ax.set_title("Plot 10 (Section 27): Latent Dimension vs Reconstruction MSE")
ax.grid(alpha=0.3)
plt.tight_layout()
save_eps(fig, "plot_extra_section27_latent_dim_vs_mse.eps")
plt.show()

In [ ]:
# ============================================================
# CELL 23 — Zip everything (10 required plots + bonus plots + all tables)
# for easy download from Colab.
# ============================================================
import shutil
shutil.make_archive("/content/experiment7_outputs", 'zip', root_dir="/content", base_dir="plots")
shutil.make_archive("/content/experiment7_tables", 'zip', root_dir="/content", base_dir="tables")

print("Plot files:")
for f in sorted(os.listdir(PLOT_DIR)):
    print(" -", f)

print("\\nTable files:")
for f in sorted(os.listdir(TABLE_DIR)):
    print(" -", f)

print("\\nDownload with:")
print("from google.colab import files")
print("files.download('/content/experiment7_outputs.zip')")
print("files.download('/content/experiment7_tables.zip')")

## Mapping of files to the report

**Required plots (Section 22, all 10, each an .eps at 600 DPI):**
1. `plot1_original_vs_reconstructed.eps`
2. `plot2_fcae_training_validation_loss.eps`
3. `plot3_fc_vs_conv_reconstruction.eps`
4. `plot4_clean_noisy_denoised.eps`
5. `plot5_noise_level_vs_quality.eps`
6. `plot6_vae_latent_space.eps`
7. `plot7_vae_generated_images.eps`
8. `plot8_latent_space_interpolation.eps`
9. `plot9_vae_training_validation_loss.eps`
10. `plot10_reconstruction_error_distribution.eps`

**Bonus plots** (needed to complete Sections 24 and 27, not part of the
numbered list of 10): `plot_extra_section24_high_error_images.eps`,
`plot_extra_section27_latent_dim_vs_mse.eps`.

**Tables** (CSV files with the exact numbers from your run):
`section9_fcae_metrics.csv`, `section11_fc_vs_conv.csv`,
`section14_noise_vs_quality.csv`, `section21_model_comparison.csv`,
`section21_vae_metrics.csv`, `section24_high_error_images.csv`,
`section25_consolidated_results.csv`, `section27_latent_dimension_study.csv`.

Every number printed above/shown in the DataFrames is what you should copy
into the empty tables in `Experiment_7.tex` (Sections 9, 11, 14, 21, 24, 25
and 27) — these values come from your own execution, as the brief requires.